<a href="https://colab.research.google.com/github/faisu6339-glitch/LLMs/blob/main/Cross_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Code Implementation Comparison: Self-Attention vs. Cross-Attention

Looking at the Python snippets for `self_attention` and `cross_attention`, the core mathematical operations (similarity calculation, scaling, softmax, weighted sum) are very similar. The fundamental difference lies in **where the Query (Q), Key (K), and Value (V) matrices originate from**.

### `cross_attention`:

```python
def cross_attention(Q, K, V):
    # Q comes from the target sequence (e.g., decoder)
    # K, V come from the source sequence (e.g., encoder)
    ...
    scores = np.matmul(Q, K.transpose(0, 2, 1))
    ...
    output = np.matmul(attention_weights, V)
    ...
```

*   **Inputs:** It explicitly takes three distinct inputs: `Q`, `K`, and `V`. `Q` represents the query from one sequence (e.g., decoder state), while `K` and `V` represent keys and values from *another* sequence (e.g., encoder output).
*   **Shapes:** The `seq_len_target` for `Q` can be different from `seq_len_source` for `K` and `V`.

### `self_attention`:

```python
def self_attention(X):
    # Q, K, V all derived from the SAME input sequence X
    Q = X
    K = X
    V = X
    ...
    scores = np.matmul(Q, K.transpose(0, 2, 1)) # Q and K here are derived from X
    ...
    output = np.matmul(attention_weights, V) # V here is derived from X
    ...
```

*   **Inputs:** It takes a single input `X` (the sequence itself). `Q`, `K`, and `V` are all set to be `X` (or would be derived from `X` via linear transformations $W_Q, W_K, W_V$ in a more complete implementation).
*   **Shapes:** Since Q, K, and V all come from the same input `X`, they inherently share the same sequence length (`seq_len`). The attention weights matrix will therefore be square (e.g., `seq_len` x `seq_len`).

**In essence, the `cross_attention` function models relationships *between* two different sequences, while the `self_attention` function models relationships *within* a single sequence, reflecting their conceptual definitions.**

Absolutely. Let's understand **self-attention and cross-attention with the same simple example**, and then implement both in Python using NumPy.

---

# 1. Self-Attention Example

Suppose our sentence is:

```text
"I love cats"
```

We represent each word with a small vector:

```text
I      → [1, 0]
love   → [0, 1]
cats   → [1, 1]
```

So our input matrix is:

$$
X =
\begin{bmatrix}
1&0\\
0&1\\
1&1
\end{bmatrix}
$$

In self-attention:

```text
              SAME INPUT
                  X
              /   |   \
             Q    K    V
              \   |   /
             Attention
```

Q, K and V all come from **the same sentence**.

---

# 2. Basic Self-Attention Program

In [4]:
import numpy as np

# Input tokens
X = np.array([
    [1, 0],   # I
    [0, 1],   # love
    [1, 1]    # cats
], dtype=float)

# Weight matrices
W_Q = np.array([
    [1, 0],
    [0, 1]
], dtype=float)

W_K = np.array([
    [1, 0],
    [0, 1]
], dtype=float)

W_V = np.array([
    [1, 0],
    [0, 1]
], dtype=float)


# Step 1: Calculate Q, K, V
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("Q:")
print(Q)

print("\nK:")
print(K)

print("\nV:")
print(V)


# Step 2: Calculate attention scores
scores = Q @ K.T

print("\nAttention Scores:")
print(scores)


# Step 3: Scale scores
d_k = K.shape[1]

scaled_scores = scores / np.sqrt(d_k)

print("\nScaled Scores:")
print(scaled_scores)


# Step 4: Softmax
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

attention_weights = softmax(scaled_scores)

print("\nAttention Weights:")
print(attention_weights)


# Step 5: Multiply weights by V
output = attention_weights @ V

print("\nSelf-Attention Output:")
print(output)

Q:
[[1. 0.]
 [0. 1.]
 [1. 1.]]

K:
[[1. 0.]
 [0. 1.]
 [1. 1.]]

V:
[[1. 0.]
 [0. 1.]
 [1. 1.]]

Attention Scores:
[[1. 0. 1.]
 [0. 1. 1.]
 [1. 1. 2.]]

Scaled Scores:
[[0.70710678 0.         0.70710678]
 [0.         0.70710678 0.70710678]
 [0.70710678 0.70710678 1.41421356]]

Attention Weights:
[[0.40111209 0.19777581 0.40111209]
 [0.19777581 0.40111209 0.40111209]
 [0.24825508 0.24825508 0.50348984]]

Self-Attention Output:
[[0.80222419 0.59888791]
 [0.59888791 0.80222419]
 [0.75174492 0.75174492]]


---

# 3. What Is Happening?

The important part is:

```python
Q = X @ W_Q
K = X @ W_K
V = X @ W_V
```

All three use:

```python
X
```

That's why this is **self-attention**.

Then:

```python
scores = Q @ K.T
```

calculates how strongly each token relates to every other token.

For example:

```text
             I      love      cats
          ┌──────┬──────┬──────┐
I         │      │      │      │
love      │      │      │      │
cats      │      │      │      │
          └──────┴──────┴──────┘
```

The attention matrix tells us:

> "For each word, how much should I pay attention to the other words?"

---

# 4. Now Cross-Attention

Let's use a translation example.

Suppose:

```text
English sentence:

"I love cats"
```

The **encoder** processes this sentence.

Its output might be:

```text
Encoder output:

I      → [1, 0]
love   → [0, 1]
cats   → [1, 1]
```

Now the decoder is generating the Hindi translation:

```text
"मैं ..."
```

The decoder has its own representation:

```text
मैं → [1, 1]
```

Now something different happens.

### Query comes from decoder:

```text
Decoder
   ↓
   Q
```

### Key and Value come from encoder:

```text
Encoder
   ↓
  K,V
```

So:

```text
             ENCODER
          I  love  cats
             ↓
            K,V
             ↓
       ┌──────────────┐
       │Cross Attention│
       └───────┬──────┘
               ↑
               Q
               │
            DECODER
              "मैं"
```

---

# 5. Cross-Attention Program

In [5]:
import numpy as np

# -----------------------------------
# ENCODER OUTPUT
# -----------------------------------

encoder_output = np.array([
    [1, 0],   # I
    [0, 1],   # love
    [1, 1]    # cats
], dtype=float)


# -----------------------------------
# DECODER INPUT
# -----------------------------------

decoder_input = np.array([
    [1, 1]    # मैं
], dtype=float)


# -----------------------------------
# Weight matrices
# -----------------------------------

W_Q = np.array([
    [1, 0],
    [0, 1]
], dtype=float)

W_K = np.array([
    [1, 0],
    [0, 1]
], dtype=float)

W_V = np.array([
    [1, 0],
    [0, 1]
], dtype=float)


# -----------------------------------
# CROSS ATTENTION
# -----------------------------------

# Query comes from DECODER
Q = decoder_input @ W_Q

# Key comes from ENCODER
K = encoder_output @ W_K

# Value comes from ENCODER
V = encoder_output @ W_V


print("Query (from Decoder):")
print(Q)

print("\nKey (from Encoder):")
print(K)

print("\nValue (from Encoder):")
print(V)


# -----------------------------------
# Attention Scores
# -----------------------------------

scores = Q @ K.T

print("\nAttention Scores:")
print(scores)


# -----------------------------------
# Scaling
# -----------------------------------

d_k = K.shape[1]

scaled_scores = scores / np.sqrt(d_k)

print("\nScaled Scores:")
print(scaled_scores)


# -----------------------------------
# Softmax
# -----------------------------------

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


attention_weights = softmax(scaled_scores)

print("\nCross-Attention Weights:")
print(attention_weights)


# -----------------------------------
# Weighted Sum
# -----------------------------------

output = attention_weights @ V

print("\nCross-Attention Output:")
print(output)

Query (from Decoder):
[[1. 1.]]

Key (from Encoder):
[[1. 0.]
 [0. 1.]
 [1. 1.]]

Value (from Encoder):
[[1. 0.]
 [0. 1.]
 [1. 1.]]

Attention Scores:
[[1. 1. 2.]]

Scaled Scores:
[[0.70710678 0.70710678 1.41421356]]

Cross-Attention Weights:
[[0.24825508 0.24825508 0.50348984]]

Cross-Attention Output:
[[0.75174492 0.75174492]]


---

# 6. Understand the Difference in the Code

This is the **most important part**.

### Self-attention

```python
Q = X @ W_Q
K = X @ W_K
V = X @ W_V
```

Everything comes from:

```python
X
```

Therefore:

```text
Q ← X
K ← X
V ← X
```

---

### Cross-attention

```python
Q = decoder_input @ W_Q

K = encoder_output @ W_K

V = encoder_output @ W_V
```

Therefore:

```text
Q ← Decoder
K ← Encoder
V ← Encoder
```

That's the fundamental difference.

---

# 7. Let's Look at the Shapes

This makes cross-attention much easier to understand.

Suppose:

```text
Encoder has 3 tokens

"I" "love" "cats"
```

and decoder currently has:

```text
"मैं"
```

Then:

```text
Encoder output
    ↓
   3 × 2

Decoder input
    ↓
   1 × 2
```

After projections:

```text
Q = 1 × 2

K = 3 × 2

V = 3 × 2
```

Now:

$$
QK^T
$$

becomes:

```text
(1 × 2) × (2 × 3)
```

giving:

```text
1 × 3
```

So we get:

```text
                  Encoder
              I     love    cats
              ↓      ↓       ↓
Attention = [0.20   0.30    0.50]
```

The decoder is saying:

```text
I      → 20%
love   → 30%
cats   → 50%
```

So the decoder focuses most on **"cats"** for this particular step.

---

# 8. Self-Attention Shapes

For:

```text
"I love cats"
```

we have:

```text
3 tokens
```

Therefore:

```text
Q = 3 × 2
K = 3 × 2
V = 3 × 2
```

Then:

$$
QK^T
$$

gives:

```text
(3 × 2) × (2 × 3)
```

result:

```text
3 × 3
```

So every token gets attention scores for every token:

```text
             I     love    cats
        ┌───────┬───────┬───────┐
I       │  ...  │  ...  │  ...  │
love    │  ...  │  ...  │  ...  │
cats    │  ...  │  ...  │  ...  │
        └───────┴───────┴───────┘
```

---

# 9. Side-by-Side Code

If you remember only one thing, remember this:

```python
# =========================
# SELF ATTENTION
# =========================

Q = X @ W_Q
K = X @ W_K
V = X @ W_V
```

versus:

```python
# =========================
# CROSS ATTENTION
# =========================

Q = decoder @ W_Q
K = encoder @ W_K
V = encoder @ W_V
```

Everything after that is basically the same:

```python
scores = Q @ K.T

scores = scores / np.sqrt(d_k)

weights = softmax(scores)

output = weights @ V
```

---

# 10. Real Transformer Picture

Think of an encoder-decoder Transformer like this:

```text
                    ENCODER
                       │
             "I love cats"
                       │
                       ↓
               Self-Attention
                       │
                       ↓
                Encoder Output
                   │       │
                   │       │
                   K       V
                    \     /
                     \   /
                      ↓ ↓
                Cross-Attention
                      ↑
                      │
                      Q
                      │
                   Decoder
                      │
                Self-Attention
                      │
                      ↓
                "मैं ..."
```

The decoder actually has **two different attention operations**:

### ① Decoder Self-Attention

```text
Decoder tokens
      ↓
    Q K V
      ↓
Self-Attention
```

It asks:

> "How are my previously generated words related?"

### ② Cross-Attention

```text
Decoder       Encoder
   ↓             ↓
   Q            K,V
    \           /
     \         /
      Cross-Attention
```

It asks:

> "Which encoder information should I use to generate the next word?"

---

## 🧠 Easy way to remember

Imagine you're a student.

**Self-attention:**

> I look at my own notes and connect different topics.

**Cross-attention:**

> I look at the textbook while writing my answer.

```text
SELF
────────────────
Same source
Q ← Same
K ← Same
V ← Same


CROSS
────────────────
Different sources
Q ← Source A
K ← Source B
V ← Source B
```

### Final formula to put in your notes

$$
\boxed{\text{Self-Attention: } Q,K,V \text{ come from the same sequence}}
$$

$$
\boxed{\text{Cross-Attention: } Q \text{ comes from one sequence, }K,V\text{ from another}}
$$

A very useful next exercise is to take the above program and **manually calculate the Q, K, V, \(QK^T\), softmax attention weights, and final output with actual numbers**. That will make the mechanism much clearer than just memorizing the formulas.

The easiest way to understand the difference is:

> **Self-attention = tokens look at other tokens in the SAME sequence.**
> **Cross-attention = tokens in one sequence look at tokens in ANOTHER sequence.**

This distinction is extremely important in **Transformers, LLMs, encoder-decoder models, and multimodal models**.

---

# 1. Self-Attention

In **self-attention**, the **Query (Q), Key (K), and Value (V)** all come from the **same sequence**.

Suppose the sentence is:

```text
"The cat drank the milk"
```

Each word looks at the other words to understand its context.

```text
                Self-Attention
                     ↓
"The"  "cat"  "drank"  "the"  "milk"
  ↕      ↕       ↕        ↕       ↕
  └──────┴───────┴────────┴───────┘
       SAME sequence
```

For example, when processing:

```text
"cat"
```

self-attention can determine that:

```text
cat → strongly related to → drank
cat → related to → the
cat → less related to → milk
```

### Mathematically

We create:

$$Q = XW_Q$$

$$K = XW_K$$

$$V = XW_V$$

Notice that **X is the same input** for all three.

Then:

$$Attention(Q,K,V)
=
softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

So:

```text
Same sequence X
      ↓
 ┌────┼────┐
 ↓    ↓    ↓
 Q    K    V
 └────┼────┘
      ↓
 Attention
```

---

# 2. Cross-Attention

In **cross-attention**, Q comes from **one sequence**, while K and V come from **another sequence**.

This is the key difference.

```text
Sequence A                 Sequence B
   ↓                           ↓
   Q                         K, V
   │                           │
   └──────── Cross ────────────┘
             Attention
```

For example, consider machine translation:

```text
English:
"I love cats"

        ↓ Encoder

Encoder representations
h1    h2    h3
I    love  cats
        ↑
        │
     K and V
        │
        │
     Decoder
        ↑
        │
      Query
        Q
```

The decoder generates the translated sentence:

```text
"J'aime les chats"
```

When generating a French word, the decoder uses **cross-attention to look at the encoder's English representations**.

---

# 3. The Biggest Difference

| Feature            | Self-Attention                             | Cross-Attention                 |
| :----------------- | :----------------------------------------- | :------------------------------ |
| Q comes from       | Same sequence                              | One sequence                    |
| K comes from       | Same sequence                              | Another sequence                |
| V comes from       | Same sequence                              | Another sequence                |
| Purpose            | Understand relationships within a sequence | Connect two sequences           |
| Example            | Word ↔ word                                | Decoder ↔ encoder               |
| Used in            | Encoder/LLM layers                         | Encoder-decoder Transformers    |
| Information source | Internal                                   | External/another representation |

The most important thing to memorize:

```text
SELF-ATTENTION

          SAME
           ↓
       ┌───────┐
       │ Input │
       └───┬───┘
       ┌───┼───┐
       ↓   ↓   ↓
       Q   K   V
```

versus:

```text
CROSS-ATTENTION

 Sequence 1          Sequence 2
     ↓                    ↓
     Q                   K,V
      \                  /
       \                /
        ↓              ↓
         Cross-Attention
```

---

# 4. Real Example

Let's say we have:

```text
English:
"How are you?"
```

and we're translating it into Hindi:

```text
"आप कैसे हैं?"
```

The **encoder** processes:

```text
How → are → you
```

and creates:

```text
h1   h2   h3
```

Now the decoder is generating:

```text
आप
```

The decoder has its own representation:

```text
decoder state → Q
```

The encoder representations provide:

```text
h1 h2 h3 → K,V
```

Therefore:

$$Q_{decoder}$$

interacts with:

$$K_{encoder}, V_{encoder}$$

This is **cross-attention**.

```text
             ENCODER
        "How are you?"
              │
          h1  h2  h3
              │
            K, V
              │
              ↓
       ┌──────────────┐
       │Cross Attention│
       └───────┬──────┘
               ↑
               │ Q
           DECODER
               │
              "आप"
```

---

# 5. Self-Attention Inside the Decoder

Here's where things get interesting.

A Transformer decoder can have **both self-attention and cross-attention**.

Suppose the decoder has already generated:

```text
"आप कैसे"
```

### First: Decoder Self-Attention

The decoder looks at its **own previously generated tokens**:

```text
आप ─── कैसे
 │      │
 └──────┘
    ↓
Self-Attention
```

So:

```text
Q = decoder
K = decoder
V = decoder
```

### Then: Cross-Attention

The decoder looks at the encoder:

```text
Decoder                    Encoder
"आप कैसे"                  "How are you?"
    ↓                           ↓
    Q                         K,V
     \                         /
      \                       /
       Cross-Attention
```

So:

```text
Decoder → Query
Encoder → Key + Value
```

This allows the decoder to ask:

> "Which parts of the English sentence are important for generating my next Hindi word?"

---

# 6. Transformer Architecture

A simplified original Transformer looks like this:

```text
                    ENCODER
                       │
             ┌─────────┴─────────┐
             │                   │
        Self-Attention       Feed Forward
             │                   │
             └─────────┬─────────┘
                       │
                Encoder Output
                       │
                 K         V
                  \       /
                   \     /
                    ↓   ↓
              CROSS-ATTENTION
                    ↑
                    │ Q
              SELF-ATTENTION
                    ↑
                Decoder
                    │
              Feed Forward
                    │
                  Output
```

The decoder therefore has two different attention mechanisms:

### Decoder Self-Attention

```text
Decoder tokens
     ↓
    Q K V
     ↓
Self-Attention
```

### Decoder Cross-Attention

```text
Decoder       Encoder
   ↓             ↓
   Q            K,V
    \           /
     \         /
   Cross-Attention
```

---

# 7. Simple Analogy

Imagine you're studying for an exam.

### Self-attention

You are reading **your own notes** and connecting different concepts.

```text
Your Notes
   ↓
Concept A ↔ Concept B ↔ Concept C
```

That's self-attention.

### Cross-attention

You're writing an answer while looking at a **textbook**.

```text
Your answer              Textbook
     ↓                      ↓
   Query                  Key/Value
        \                  /
         \                /
          Cross-Attention
```

You're using information from another source.

---

# 8. What About LLMs Like GPT?

This is an important point.

**GPT-style decoder-only LLMs primarily use masked/casual self-attention**, not the classic encoder-decoder cross-attention used in models such as the original Transformer translation architecture.

For example:

```text
"I am learning"
```

GPT's self-attention allows:

```text
"I" ───────┐
           ↓
"am" ───→ "learning"
           ↑
```

Each token attends to the relevant previous tokens.

There isn't a separate encoder sequence supplying K/V in the standard GPT architecture.

---

# 9. Where Cross-Attention Is Commonly Used

Cross-attention is especially important when **two different information sources need to interact**.

### Machine Translation

```text
English Encoder
      ↓
     K,V
      ↓
Cross-Attention ← Q ← Decoder
```

### Image Captioning

```text
Image features
     ↓
    K,V
     ↓
Cross-Attention
     ↑
     Q
     │
Text Decoder
```

The decoder can effectively ask:

> "Which part of the image should I focus on to generate the next word?"

### Multimodal AI

For example:

```text
Image ──→ K,V
            \
             Cross-Attention
            /
Text ────→ Q
```

This allows text representations to interact with image representations.

---

# 10. One Formula Makes the Difference Crystal Clear

### Self-Attention

$$Q=XW_Q$$

$$K=XW_K$$

$$V=XW_V$$

Therefore:

$$Attention=
softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

All come from **X**.

---

### Cross-Attention

Suppose:

*   Decoder input = \(X_d\)
*   Encoder output = \(X_e\)

Then:

$$Q=X_dW_Q$$

but:

$$K=X_eW_K$$

$$V=X_eW_V$$

Therefore:

$$CrossAttention=
softmax\left(
\frac{Q_{decoder}K_{encoder}^T}
{\sqrt{d_k}}
\right)V_{encoder}$$

So the fundamental difference is:

```text
SELF:

Q ← Sequence A
K ← Sequence A
V ← Sequence A


CROSS:

Q ← Sequence A
K ← Sequence B
V ← Sequence B
```

## ⭐ One-line exam definition

> **Self-attention allows tokens within the same sequence to attend to one another, whereas cross-attention allows tokens from one sequence to attend to representations from another sequence.**

If you're learning **attention mechanisms for LLMs**, the best next step is to understand **how Q, K, V matrices are actually calculated numerically in self-attention and cross-attention**, including a small Python program.

In [1]:
import numpy as np

def cross_attention(Q, K, V):
    """
    Implements the cross-attention mechanism.

    Args:
        Q (numpy.ndarray): Query matrix (from target sequence).
        K (numpy.ndarray): Key matrix (from source sequence).
        V (numpy.ndarray): Value matrix (from source sequence).

    Returns:
        numpy.ndarray: The output of the cross-attention mechanism.
        numpy.ndarray: The attention weights.
    """
    # 1. Similarity Calculation (Query-Key Interaction)
    #   Q: (batch_size, seq_len_target, d_k)
    #   K: (batch_size, seq_len_source, d_k)
    #   K.T: (batch_size, d_k, seq_len_source)
    scores = np.matmul(Q, K.transpose(0, 2, 1)) # (batch_size, seq_len_target, seq_len_source)

    # 2. Scaling
    d_k = Q.shape[-1] # Dimension of keys
    scaled_scores = scores / np.sqrt(d_k)

    # 3. Softmax Application
    # Apply softmax across the source sequence length dimension (last dimension)
    attention_weights = np.exp(scaled_scores - np.max(scaled_scores, axis=-1, keepdims=True))
    attention_weights = attention_weights / np.sum(attention_weights, axis=-1, keepdims=True)

    # 4. Weighted Sum of Values
    #   attention_weights: (batch_size, seq_len_target, seq_len_source)
    #   V: (batch_size, seq_len_source, d_v)
    output = np.matmul(attention_weights, V) # (batch_size, seq_len_target, d_v)

    return output, attention_weights

# --- Demo of Cross-Attention ---

# Define dimensions
batch_size = 1
seq_len_target = 3 # e.g., 3 tokens in decoder output
seq_len_source = 5 # e.g., 5 tokens in encoder output
d_model = 64       # Embedding dimension

# Simulate Query, Key, Value matrices
# Q from decoder side (e.g., current decoder state)
Q = np.random.rand(batch_size, seq_len_target, d_model)
# K, V from encoder side (e.g., encoder outputs)
K = np.random.rand(batch_size, seq_len_source, d_model)
V = np.random.rand(batch_size, seq_len_source, d_model)

print("Input Query (Q) shape:", Q.shape)
print("Input Key (K) shape:", K.shape)
print("Input Value (V) shape:", V.shape)

# Perform cross-attention
attention_output, weights = cross_attention(Q, K, V)

print("\nCross-Attention Output shape:", attention_output.shape)
print("Attention Weights shape:", weights.shape)

print("\nFirst query's attention weights (sum to 1 across source sequence):")
print(weights[0, 0, :])
print("Sum of first query's attention weights:", np.sum(weights[0, 0, :]))

print("\nExample of attention output (first target token's contextualized representation):")
print(attention_output[0, 0, :5]) # Display first 5 elements for brevity

Input Query (Q) shape: (1, 3, 64)
Input Key (K) shape: (1, 5, 64)
Input Value (V) shape: (1, 5, 64)

Cross-Attention Output shape: (1, 3, 64)
Attention Weights shape: (1, 3, 5)

First query's attention weights (sum to 1 across source sequence):
[0.1889924  0.30383641 0.14959691 0.2396908  0.11788347]
Sum of first query's attention weights: 1.0

Example of attention output (first target token's contextualized representation):
[0.61263754 0.68269337 0.54503616 0.31741461 0.71057386]


In [2]:
import numpy as np

def self_attention(X):
    """
    Implements the self-attention mechanism.

    Args:
        X (numpy.ndarray): Input sequence (batch_size, seq_len, d_model).

    Returns:
        numpy.ndarray: The output of the self-attention mechanism.
        numpy.ndarray: The attention weights.
    """
    # Assume W_Q, W_K, W_V are identity matrices for simplicity in this demo
    # In a real scenario, these would be learnable weight matrices.
    Q = X # In self-attention, Q, K, V come from the same input X
    K = X
    V = X

    # 1. Similarity Calculation (Query-Key Interaction)
    #   Q: (batch_size, seq_len, d_model)
    #   K: (batch_size, seq_len, d_model)
    #   K.T: (batch_size, d_model, seq_len)
    scores = np.matmul(Q, K.transpose(0, 2, 1)) # (batch_size, seq_len, seq_len)

    # 2. Scaling
    d_k = Q.shape[-1] # Dimension of keys (which is d_model here)
    scaled_scores = scores / np.sqrt(d_k)

    # 3. Softmax Application
    # Apply softmax across the sequence length dimension (last dimension)
    attention_weights = np.exp(scaled_scores - np.max(scaled_scores, axis=-1, keepdims=True))
    attention_weights = attention_weights / np.sum(attention_weights, axis=-1, keepdims=True)

    # 4. Weighted Sum of Values
    #   attention_weights: (batch_size, seq_len, seq_len)
    #   V: (batch_size, seq_len, d_model)
    output = np.matmul(attention_weights, V) # (batch_size, seq_len, d_model)

    return output, attention_weights

# --- Demo of Self-Attention ---

# Define dimensions
batch_size = 1
seq_len = 5       # e.g., 5 tokens in a sentence
d_model = 64      # Embedding dimension

# Simulate Input sequence X
X = np.random.rand(batch_size, seq_len, d_model)

print("\nInput X shape for self-attention:", X.shape)

# Perform self-attention
self_attention_output, self_weights = self_attention(X)

print("Self-Attention Output shape:", self_attention_output.shape)
print("Self-Attention Weights shape:", self_weights.shape)

print("\nFirst token's self-attention weights (sum to 1 across sequence):")
print(self_weights[0, 0, :])
print("Sum of first token's self-attention weights:", np.sum(self_weights[0, 0, :]))



Input X shape for self-attention: (1, 5, 64)
Self-Attention Output shape: (1, 5, 64)
Self-Attention Weights shape: (1, 5, 5)

First token's self-attention weights (sum to 1 across sequence):
[0.41040286 0.14335772 0.13787595 0.15893277 0.14943071]
Sum of first token's self-attention weights: 1.0


# Self-Attention Program